# 10주차 ① 시퀀스 데이터와 임베딩 — 실습 1~3

**목표**: 원-핫의 두 가지 문제를 확인하고, `nn.Embedding` 이 **학습되는 층**임을 `.grad` 로 직접 보며,
패딩된 배치의 **`(batch, seq_len, embed_dim)`** 세 축을 읽는다.

> **준비물**: 배포받은 `preprocess_text.py` 와 `data/nsmc_subset/`.

```
   5~9주차 : 이미지          "옆 픽셀과의 관계"가 중요했다 → CNN
   10주차  : 텍스트          "앞뒤 단어의 순서"가 중요하다 → ?

   "이 영화 정말 재미없다"        vs        "재미없다 정말 영화 이"
        의미가 있다                            뜻이 무너진다
```

> **핵심 메시지 ★**: 9주차에 쓴 **ViT 의 'T' 가 트랜스포머**입니다.
> 그 안에 들어 있는 것이 오늘 배우는 **어텐션**이고, **다음 주에 그걸 직접 짭니다.**
> **가져다 쓴 것의 안을 여는 순서**입니다 — 4주차에 `.fit()` 을 열었던 것과 같습니다.

> **1·2교시는 3교시를 위한 준비입니다.** 오늘의 진짜 목적은 LSTM 을 잘 만드는 게 아니라,
> **어텐션이 왜 필요한지 몸으로 느끼는 것**입니다.

### 원-핫의 두 가지 문제 ★★

```
   어휘사전 : {영화:0, 정말:1, 재미없다:2, 좋다:3, ...}

   좋다     [0,1,0,0,0]
   훌륭하다 [0,0,0,1,0]      →  두 벡터의 내적 = 0.  아무 관계도 없다고 표현된다
   최악     [0,0,0,0,1]      →  이것도 0.  "좋다-훌륭하다" 와 "좋다-최악" 이 같다?!
```

| 문제 | 설명 |
|---|---|
| **① 차원이 너무 크다** | 어휘가 5만이면 단어 하나가 **5만 차원**. 그중 하나만 1 |
| **② 단어 사이 관계가 0** ★ | "좋다"와 "훌륭하다"의 유사도 = 0. **전부 똑같이 무관** |

> **②가 훨씬 심각합니다.** 모델이 *"좋다와 훌륭하다는 비슷하다"* 를 **처음부터 다시 배워야** 하니까요.
> 해법은 **조밀한 벡터를 학습하는 것** — 그게 **임베딩**입니다.
> 1주차의 *"특징을 사람이 설계하지 않는다"* 가 또 나옵니다.

## 실습 1 — `nn.Embedding` 동작 확인 ★

In [ ]:
# 셀 1 — 임베딩은 무엇을 하나
import torch, torch.nn as nn

VOCAB, DIM = 10, 4                      # 어휘 10개, 임베딩 4차원 (작게 해서 눈으로 본다)
torch.manual_seed(0)
emb = nn.Embedding(VOCAB, DIM)

print("가중치 shape :", emb.weight.shape)      # (10, 4) — 단어마다 한 줄
print(emb.weight)

In [ ]:
# 셀 2 — 인덱스를 넣으면 그 줄이 나온다
idx = torch.tensor([2, 5, 2])           # 2번, 5번, 다시 2번 단어
out = emb(idx)
print("입력 :", idx, "→ 출력 shape :", out.shape)      # (3, 4)
print(out)
print("\n2번이 두 번 나왔는데 같은 벡터인가? :", torch.equal(out[0], out[2]))

> **관찰 포인트 ★**: `nn.Embedding` 은 **조회표(lookup table)** 처럼 동작합니다.
> 인덱스를 넣으면 그 번호의 **줄을 꺼내** 줍니다. 같은 단어는 항상 같은 벡터입니다.

In [ ]:
# 셀 3 — 그런데 "학습되는" 조회표다  ★
emb = nn.Embedding(VOCAB, DIM)

out = emb(torch.tensor([2]))
loss = out.sum()                        # 아무 손실이나
loss.backward()

print("2번 단어의 기울기 :", emb.weight.grad[2])
print("0번 단어의 기울기 :", emb.weight.grad[0], "  ← 안 쓴 단어는 0")
print("\nrequires_grad :", emb.weight.requires_grad)
print("기울기가 0이 아닌 단어 수 :", (emb.weight.grad.abs().sum(1) > 0).sum().item(), "/", VOCAB)

> **핵심 메시지 ★★ (출제 지점)**: **`nn.Embedding` 은 고정된 조회표가 아니라 학습되는 층**입니다.
> `emb.weight` 는 **파라미터**이고 `.grad` 가 채워집니다.
> 학습이 진행되면 **비슷한 문맥에 나오는 단어끼리 벡터가 가까워집니다.**

> **관찰 포인트**: **배치에 나온 단어의 기울기만** 채워지고 나머지는 0 입니다.
> 그래서 어휘가 커도 학습이 무겁지 않습니다.

In [ ]:
# 셀 4 — 원-핫 × 행렬 = 임베딩 조회  (수학적으로 같다는 확인)
V, D = 6, 3
torch.manual_seed(0)
W = torch.randn(V, D)

onehot = torch.zeros(1, V); onehot[0, 4] = 1.0
print("원-핫 × W :", (onehot @ W).squeeze())
print("W[4]      :", W[4])
print("→ 같은 값입니다. 임베딩은 이 곱셈을 '조회'로 대신할 뿐입니다\n")

V, D = 20000, 128
print(f"원-핫 입력 → Linear(20000, 128) : {V*D:,} 개")
print(f"nn.Embedding(20000, 128)        : {V*D:,} 개  ← 수는 같다")
print("\n차이는 '계산 방식'이다:")
print("  원-핫: 20000차원 벡터 × 행렬 곱  →  대부분 0과의 곱셈, 낭비")
print("  임베딩: 인덱스로 한 줄 조회      →  즉시")

## 실습 2 — 전처리 코드 **읽기** (제공)

> **짜지 마세요. 읽고 흐름만 파악합니다.** 토큰화·어휘사전에 시간을 쓰면
> 3교시 어텐션에 도달하지 못합니다. 이 주차의 목적은 어텐션입니다.

```
   ① 원문        "이 영화 정말 재미없다"
        ↓ 토큰화
   ② 토큰        ["이", "영화", "정말", "재미없다"]
        ↓ 어휘사전 (빈도순, 상위 20,000개)
   ③ 인덱스      [3, 27, 15, 842]
        ↓ 패딩/자르기 (최대 길이 40)
   ④ 고정 길이   [3, 27, 15, 842, 0, 0, ..., 0]      ← 0 = <pad>
```

| 특수 토큰 | 뜻 |
|---|---|
| `<pad>` (0) | 길이를 맞추기 위한 빈칸 |
| `<unk>` (1) | 어휘사전에 없는 단어 |

In [ ]:
# 셀 5 — 제공 코드를 불러 쓴다
from preprocess_text import load_nsmc, build_vocab, encode, tokenize, save_vocab, real_length

train_texts, train_labels, val_texts, val_labels = load_nsmc("data/nsmc_subset")
print(f"훈련 {len(train_texts):,}건 / 검증 {len(val_texts):,}건")

vocab = build_vocab(train_texts, max_size=20000)       # ★ 크기 고정 (배포본과 동일해야 함)
print("어휘 크기 :", len(vocab))
print("예시 :", list(vocab.items())[:10])

print("\n원문   :", train_texts[0])
print("토큰   :", tokenize(train_texts[0]))
print("인덱스 :", encode(train_texts[0], vocab, max_len=40))

> ⚠️ **어휘 크기(20,000)와 최대 길이(40)는 고정입니다.**
> 학생마다 다르면 **3교시 실습 6의 성능 비교가 성립하지 않습니다.**

> **핵심 메시지 ★**: 어휘사전은 **훈련 데이터로만** 만듭니다.
> 검증·테스트 문장에 있는 단어까지 넣으면 **정보가 새어 나갑니다.**
> 6주차에 *"테스트로 설정을 고르면 안 된다"* 고 한 것과 같은 이유입니다.
> 검증에서 처음 보는 단어는 `<unk>` 로 처리됩니다.

| 함정 | 설명 |
|---|---|
| `<pad>` 를 0번에 두는 이유 | `nn.Embedding(..., padding_idx=0)` 으로 **학습에서 제외**할 수 있다 |
| 어휘사전을 저장 안 함 | 다음에 모델을 불러와도 **인덱스가 달라져** 못 쓴다. 반드시 함께 저장 |
| 최대 길이가 너무 짧음 | 뒷부분이 잘린다. **감성은 대개 문장 끝에** 있어서 치명적 |

In [ ]:
# 셀 6 — 어휘사전은 반드시 저장한다  ★ 2·3교시에서 같은 것을 쓴다
save_vocab(vocab, "models/vocab.json")
print("저장 완료 : models/vocab.json")

# 최대 길이 40 이 적절한지 직접 확인
lens = [real_length(encode(t, vocab, 200)) for t in train_texts[:3000]]
over = sum(1 for l in lens if l > 40)
print(f"\n앞 3,000건 | 평균 {sum(lens)/len(lens):.1f} 토큰 / 최대 {max(lens)}")
print(f"max_len=40 에서 잘리는 문장 : {over/len(lens)*100:.1f}%")

## 실습 3 — 패딩·배치 shape 확인

In [ ]:
# 셀 7 — 배치를 만들고 shape 을 읽는다
from torch.utils.data import TensorDataset, DataLoader

MAX_LEN, VOCAB_SIZE, EMBED_DIM = 40, len(vocab), 128

X = torch.tensor([encode(t, vocab, MAX_LEN) for t in train_texts])
y = torch.tensor(train_labels)
print("전체 X shape :", X.shape)                 # (2만, 40)

loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print("배치 x :", xb.shape, "| 배치 y :", yb.shape)     # (64, 40) / (64,)

emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
e = emb(xb)
print("임베딩 후 :", e.shape)                    # ★ (64, 40, 128)

> **핵심 메시지 ★★ (출제 지점)**: **`(batch, seq_len, embed_dim)`** — 세 축의 뜻을 말할 수 있어야 합니다.
> ```
>   64  : 한 번에 처리하는 문장 수
>   40  : 문장 하나의 토큰 수 (패딩 포함, 고정)
>   128 : 토큰 하나를 나타내는 벡터의 길이
> ```
> 7주차 CNN 에서 `(batch, channel, H, W)` 를 읽었던 것과 같은 훈련입니다.
> **이 3차원 텐서가 오늘 내내 나옵니다.**

In [ ]:
# 셀 8 — 패딩이 어떻게 생겼나
print("첫 문장의 인덱스 :", xb[0].tolist())
print("실제 길이(0이 아닌 것) :", (xb[0] != 0).sum().item(), "/ 40")

print("\npadding_idx=0 이면 <pad> 벡터는 전부 0 :")
print(emb.weight[0])

# 배치 전체에서 패딩이 차지하는 비율
pad_ratio = (xb == 0).float().mean().item()
print(f"\n이 배치의 {pad_ratio*100:.1f}% 가 패딩입니다")

> **관찰 포인트 ★**: `<pad>` 자리의 임베딩이 **전부 0** 입니다.
> `padding_idx=0` 이 그렇게 만들고, **이 벡터는 학습되지 않습니다.**

> **핵심 질문 (3교시 복선) ★**: 배치의 절반 이상이 패딩입니다.
> LSTM 이 이걸 다 읽고 **마지막 상태**를 쓰면, 그 마지막은 **패딩을 읽은 결과**입니다.
> **문제가 있어 보이지 않나요?** 3교시에 이 이야기로 돌아옵니다.

---

### 이 노트북 체크리스트

- [ ] 원-핫의 두 가지 문제를 말할 수 있다 ★★
- [ ] `nn.Embedding` 이 **학습되는 층**이라는 것을 `.grad` 로 확인했다 ★
- [ ] 사용된 단어의 기울기만 채워지는 것을 봤다
- [ ] 전처리 네 단계(토큰화 → 어휘사전 → 인덱스 → 패딩)를 설명할 수 있다
- [ ] 어휘사전을 훈련 데이터로만 만드는 이유를 안다
- [ ] 어휘사전을 `models/vocab.json` 으로 저장했다
- [ ] `(64, 40, 128)` 의 세 축이 무엇인지 말할 수 있다 ★
- [ ] `padding_idx` 의 역할을 안다